# 03 Feature Review

Use this after feature extraction. It checks feature table integrity before any ML baseline is trusted.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/home/ubuntu/nishn_workspce/test_pdfs_generic/.covid_audio_btp_private/covid_audio_btp")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from covid_audio_btp.notebook_utils import (
    PROJECT_ROOT,
    artifact,
    check_artifacts,
    count_table,
    read_csv,
    read_optional_csv,
    require_artifacts,
    save_table,
    value_counts_frame,
    assert_no_participant_leakage,
    assert_binary_labels_present,
    stop_if_validation_errors,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
print(PROJECT_ROOT)


In [ ]:
require_artifacts(["data/processed/features_mfcc.csv", "data/processed/spectrogram_index.csv"])
features = read_csv("data/processed/features_mfcc.csv", n=5)
spec_index = read_csv("data/processed/spectrogram_index.csv", n=5)


## Shape And Coverage Gates


In [ ]:
id_cols = {"recording_id", "participant_id", "dataset", "modality", "submodality", "label_binary", "split", "segmentation_method"}
numeric_cols = [c for c in features.columns if c not in id_cols and pd.api.types.is_numeric_dtype(features[c])]
print(f"feature rows: {len(features)}")
print(f"numeric feature columns: {len(numeric_cols)}")
if len(features) == 0:
    raise AssertionError("Feature table is empty.")
if len(numeric_cols) < 40:
    raise AssertionError("Too few numeric features. Feature extraction likely failed.")
assert_binary_labels_present(features)
assert_no_participant_leakage(features)
display(count_table(features, ["split", "modality", "label_binary"]))


## Missing, Infinite, And Constant Features


In [ ]:
x = features[numeric_cols]
x_array = x.to_numpy(dtype=float)
feature_health = pd.DataFrame({
    "feature": numeric_cols,
    "missing_rate": x.isna().mean().values,
    "infinite_rate": np.isinf(x_array).mean(axis=0),
    "std": x.replace([np.inf, -np.inf], np.nan).std(numeric_only=True).values,
})
feature_health["is_constant"] = feature_health["std"].fillna(0) == 0
save_table(feature_health.sort_values(["missing_rate", "is_constant"], ascending=False), "reports/tables/nb03_feature_health.csv")
display(feature_health.sort_values(["missing_rate", "is_constant"], ascending=False).head(30))

if (feature_health["missing_rate"] > 0.20).any():
    print("Warning: some features have more than 20% missing values.")
if feature_health["is_constant"].mean() > 0.20:
    raise AssertionError("Too many constant features. Inspect preprocessing output.")


## Feature Distributions And Correlation


In [ ]:
key_features = [c for c in ["rms_mean", "zcr_mean", "spectral_centroid_mean", "spectral_bandwidth_mean", "spectral_rolloff_mean", "duration_sec", "active_audio_ratio"] if c in features.columns]
if key_features:
    display(features.groupby(["modality", "label_binary"])[key_features].median().reset_index())
    fig, axes = plt.subplots(1, min(3, len(key_features)), figsize=(14, 4))
    if len(key_features) == 1:
        axes = [axes]
    for ax, col in zip(axes, key_features[:3]):
        sns.boxplot(data=features, x="modality", y=col, hue="label_binary", ax=ax)
        ax.set_title(col)
    fig.tight_layout()
    fig.savefig(PROJECT_ROOT / "reports/figures/nb03_key_feature_distributions.png", dpi=160)
    plt.show()

corr_cols = numeric_cols[:80]
corr = features[corr_cols].corr().abs()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="mako", cbar=True)
plt.title("Absolute feature correlation, first 80 numeric columns")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports/figures/nb03_feature_correlation_first80.png", dpi=160)
plt.show()


## Spectrogram Index Review


In [ ]:
display(count_table(spec_index, ["split", "modality", "label_binary"]))
missing_spec_paths = spec_index[~spec_index["spectrogram_path"].map(lambda p: (Path(p).exists() or (PROJECT_ROOT / p).exists()) if isinstance(p, str) else False)] if "spectrogram_path" in spec_index else pd.DataFrame()
print(f"missing spectrogram files: {len(missing_spec_paths)}")
if len(missing_spec_paths):
    display(missing_spec_paths.head(20))
    raise AssertionError("Spectrogram index points to missing files.")
